## LOAD SAPS Shapefile

In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
station_data = gpd.read_file("../data/raw/saps/Police_bounds.shp")

station_data.head()




,COMPNT_NM,CREATE_DT,VERSION,geometry
0,BOTSHABELO,20251205,1.4.2,"POLYGON ((26.77137 -29.21403, 26.7733 -29.2217..."
1,KHUBUSIDRIFT,20251205,1.4.2,"POLYGON ((27.72842 -32.53051, 27.72842 -32.531..."
2,STUTTERHEIM,20251205,1.4.2,"POLYGON ((27.50201 -32.44217, 27.49884 -32.465..."
3,MOTHERWELL,20251205,1.4.2,"POLYGON ((25.61061 -33.81772, 25.60713 -33.822..."
4,KWADWESI,20251205,1.4.2,"POLYGON ((25.45586 -33.83107, 25.4666 -33.8334..."


In [3]:
filtered = station_data[station_data["COMPNT_NM"] == "ABERDEEN"]
filtered.head()

,COMPNT_NM,CREATE_DT,VERSION,geometry
988,ABERDEEN,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359..."


## Load crime stats excel

In [4]:
crime_data = pd.read_excel("../data/raw/saps/2025-2026_-_3rd_Quarter_WEB.xlsx", sheet_name="RAW Data", header=2)
crime_data.columns = (
    crime_data.columns
    .str.strip()
    .str.replace("\n", " ")
    .str.replace(" ", "_")
)
crime_data.head()


,Crime_Category_National_contribution_placement,Crime_Category_Provincial_contribution_placement,Comp_level,Station_Crime_Category,Station,District,Province,Crime_Category,Code,NaN,...,NaN,NaN,October_2025_to__December_2025,National_contribution_placement,National_count_diff_placement,Provincial_contribution_placement,Provincial_count_diff_placement,Count_direction,Count_offence_group,No
0,Murder Station 595,Eastern Cape Murder Station 109,Station,Aberdeen Murder,Aberdeen,Sarah Baartman District,Eastern Cape,Murder,1,1.0,...,2.0,0.0,2.0,595.0,448.0,109.0,83.0,Stabilized,17 Community reported serious Crime,7085.0
1,Attempted murder Station 864,Eastern Cape Attempted murder Station 128,Station,Aberdeen Attempted murder,Aberdeen,Sarah Baartman District,Eastern Cape,Attempted murder,2,0.0,...,0.0,0.0,0.0,864.0,555.0,128.0,84.0,Stabilized,17 Community reported serious crime,7086.0
2,Culpable homicide Station 830,Eastern Cape Culpable homicide Station 120,Station,Aberdeen Culpable homicide,Aberdeen,Sarah Baartman District,Eastern Cape,Culpable homicide,3,1.0,...,0.0,0.0,1.0,830.0,1141.0,120.0,197.0,Decreased,0,7087.0
3,Robbery with aggravating circumstances Station...,Eastern Cape Robbery with aggravating circumst...,Station,Aberdeen Robbery with aggravating circumstances,Aberdeen,Sarah Baartman District,Eastern Cape,Robbery with aggravating circumstances,4,0.0,...,0.0,0.0,0.0,1141.0,721.0,191.0,135.0,Decreased,17 Community reported serious crime,7088.0
4,Common robbery Station 664,Eastern Cape Common robbery Station 77,Station,Aberdeen Common robbery,Aberdeen,Sarah Baartman District,Eastern Cape,Common robbery,6,0.0,...,0.0,1.0,2.0,664.0,416.0,77.0,67.0,Stabilized,17 Community reported serious crime,7089.0


#### Helper function to help get latest quarterly data

In [6]:
import re
from datetime import datetime

def get_latest_quarter_column(df):
    months = ['January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']
    month_pattern = '|'.join(months)

    pattern = re.compile(
        rf'(?P<start_month>{month_pattern})_(?P<start_year>\d{{4}})_to_+?(?P<end_month>{month_pattern})_(?P<end_year>\d{{4}})',
        re.IGNORECASE
    )

    latest_col = None
    latest_end_date = None

    for col in df.columns:
        col_str = str(col)
        match = pattern.search(col_str)
        if match:
            end_month = match.group('end_month').capitalize()
            end_year = int(match.group('end_year'))

            try:
                end_date = datetime.strptime(f"{end_month} {end_year}", "%B %Y")
            except ValueError:
                continue

            if latest_end_date is None or end_date > latest_end_date:
                latest_end_date = end_date
                latest_col = col
                
    if latest_col is None:
        raise ValueError("No quarterly column found matching pattern 'Month_YYYY_to_Month_YYYY'")

    return latest_col


In [9]:
crime_data["Station"] = crime_data["Station"].astype(str)
crime_data = crime_data[crime_data["Station"].str.contains("[A-Za-z]", na=False)]
crime_data["Station"].unique()
crime_data.head()

latest_quarter_col = get_latest_quarter_column(crime_data)
count_col = latest_quarter_col

# count_col = [c for c in crime_data.columns if 'October_2025' in str(c)][0]
crime_data_clean = crime_data[['Station', 'District', 'Crime_Category', count_col, 'National_contribution_placement', 'Provincial_contribution_placement', 'Count_direction']].copy()
crime_data_clean.columns = ['Station', 'District', 'Crime_Type', 'Crime_Count', 'National_placement', 'Provincial_placement', 'Count_direction']
crime_data_clean['Station'] = crime_data_clean['Station'].str.strip().str.upper()


# X = crime_data_clean.groupby('Crime_Type')['Crime_Count'].sum().sort_values(ascending=False)
# X.head()

### Looking at the Unique Crimes From Crime_Type Column To spot crimes to remove

In [10]:
unique_crimes = sorted(crime_data_clean['Crime_Type'].dropna().unique())

for crime in unique_crimes:
    print(crime)
    

17 Community reported serious Crime
Abduction
All theft not mentioned elsewhere
Arson
Assault with the intent to inflict grievous bodily harm
Attempted murder
Attempted sexual offences
Bank robbery
Burglary at non-residential premises
Burglary at residential premises
Carjacking
Commercial crime
Common assault
Common robbery
Contact crime (Crimes against the person)
Contact sexual offences
Contact-related Crime
Crime detected as a result of police action
Crimen injuria
Culpable homicide
Driving under the influence of alcohol or drugs
Drug-related crime
Illegal possession of firearms and ammunition
Kidnapping
Malicious damage to property
Murder
Neglect and ill-treatment of children
Other serious Crime
Property-related Crime
Public violence
Rape
Robbery at non-residential premises
Robbery at residential premises
Robbery of cash in transit
Robbery with aggravating circumstances
Sexual assault
Sexual offences
Sexual offences detected as a result of police action
Shoplifting
Stock-theft
TRIO

In [11]:
categories_to_remove = [
    '17 Community reported serious Crime',
    'Contact crime (Crimes against the person)',
    'Contact-related Crime',
    'Property-related Crime',
    'Other serious Crime',
    'TRIO Crime',
    'Crime detected as a result of police action',
    'Sexual offences',
    'Drug-related crime'
]

In [12]:
crime_data_clean = crime_data_clean[
    ~crime_data_clean['Crime_Type'].isin(categories_to_remove)
]
crime_data_clean.head()
crime_data_clean = crime_data_clean.dropna(subset=['Crime_Type'])
# print(crime_data_clean['Crime_Type'].unique())
crime_data_clean.shape

(43330, 7)

In [13]:

for crime in crime_data_clean['Crime_Type'].unique():
    print(crime)

Murder
Attempted murder
Culpable homicide
Robbery with aggravating circumstances
Common robbery
Public violence
Rape
Sexual assault
Crimen injuria
Neglect and ill-treatment of children
Kidnapping
Abduction
Assault with the intent to inflict grievous bodily harm
Common assault
Burglary at non-residential premises
Burglary at residential premises
Stock-theft
Shoplifting
Theft of motor vehicle and motorcycle
Theft out of or from motor vehicle
All theft not mentioned elsewhere
Arson
Malicious damage to property
Commercial crime
Driving under the influence of alcohol or drugs
Illegal possession of firearms and ammunition
Carjacking
Truck hijacking
Robbery of cash in transit
Bank robbery
Robbery at residential premises
Robbery at non-residential premises
Attempted sexual offences
Contact sexual offences
Sexual offences detected as a result of police action


### Test precision of Nominatim

In [14]:
from geopy.geocoders import Nominatim



geolocator = Nominatim(user_agent="kaggle_learn", timeout=10)
location = geolocator.geocode("Kayamandi Secondary School")

print(location.point)
print(location.address)
point = location.point
print("Latitude:", point.latitude)
print("Longitude:", point.longitude)

    

33 55m 24.0208s S, 18 50m 54.3642s E
Kayamandi Secondary School, School Crescent, Stellenbosch Ward 12, Stellenbosch, Stellenbosch Local Municipality, Cape Winelands District Municipality, Western Cape, 7599, South Africa
Latitude: -33.9233391
Longitude: 18.8484345


### Test extraction of files from SAPS unstable website

In [12]:
import requests, certifi
from bs4 import BeautifulSoup
import urllib3
import os

url = "https://www.saps.gov.za/services/crimestats.php"
DOWNLOAD_DIR = r"C:\Users\DELL\Downloads"

# Disable ssl verification
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

response = requests.get(url, verify=False)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')





table = soup.find('table', class_='table')
if not table:
    raise ValueError("Table not found on page")

rows = table.find_all('tr')

data_row = rows[1] if rows else []
cols = data_row.find_all('td')
spreadsheet_download_tag = cols[-1].find("a")
spreadsheet_download_link = spreadsheet_download_tag.get('href') if spreadsheet_download_tag else None

if not spreadsheet_download_link:
    raise ValueError("No link to download")
file_url = "https://www.saps.gov.za/services/" + spreadsheet_download_link
filename = os.path.basename(spreadsheet_download_link)
file_path = os.path.join(DOWNLOAD_DIR, filename)



In [13]:
response = requests.get(file_url, verify=False)

with open(file_path, "wb") as f:
    f.write(response.content)

print(f"Saved to: {file_path}")

Saved to: C:\Users\DELL\Downloads\2025-2026_-_3rd_Quarter_WEB.xlsx


### Merge data from the shapefile with that from the excel file on Station name 

In [15]:
crime_data_clean = crime_data_clean.rename(columns={"Station": "Station_name"})
station_data = station_data.rename(columns={"COMPNT_NM": "Station_name"})
merged = crime_data_clean.merge(station_data, on="Station_name", how="left")
merged.head()

,Station_name,District,Crime_Type,Crime_Count,National_placement,Provincial_placement,Count_direction,CREATE_DT,VERSION,geometry
0,ABERDEEN,Sarah Baartman District,Murder,2.0,595.0,109.0,Stabilized,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359..."
1,ABERDEEN,Sarah Baartman District,Attempted murder,0.0,864.0,128.0,Stabilized,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359..."
2,ABERDEEN,Sarah Baartman District,Culpable homicide,1.0,830.0,120.0,Decreased,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359..."
3,ABERDEEN,Sarah Baartman District,Robbery with aggravating circumstances,0.0,1141.0,191.0,Decreased,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359..."
4,ABERDEEN,Sarah Baartman District,Common robbery,2.0,664.0,77.0,Stabilized,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359..."


### Categorize data for less chaos 

In [16]:
Violent_Crime = ["Murder", "Attempted murder", "Culpable homicide", "Assault with the intent to inflict grievous bodily harm", "Common assault",
"Robbery with aggravating circumstances", "Common robbery",
"Carjacking", "Truck hijacking", "Robbery of cash in transit",
"Bank robbery", "Robbery at residential premises", "Robbery at non-residential premises", "Public violence"]
Sexual_Offences = ["Rape", "Sexual assault", "Attempted sexual offences", "Contact sexual offences", "Sexual offences detected as a result of police action"]
Theft_and_Burglary = ["Burglary at residential premises", "Burglary at non-residential premises", "Shoplifting", "Stock-theft", "Theft of motor vehicle and motorcycle", "Theft out of or from motor vehicle", "All theft not mentioned elsewhere"]
Crimes_against_Children = ["Neglect and ill-treatment of children", "Kidnapping", "Abduction"]
Property_Damage = ["Arson", "Malicious damage to property"]
Commercial_or_Financial = ["Commercial crime"]
Police_Action = ["Driving under the influence of alcohol or drugs", "Illegal possession of firearms and ammunition"]
Social_Crimes = ["Crimen injuria"]

In [17]:
crime_groups = {
    "Violent Crime": Violent_Crime,
    "Sexual Offences": Sexual_Offences,
    "Theft and Burglary": Theft_and_Burglary,
    "Crimes Against Children": Crimes_against_Children,
    "Property Damage": Property_Damage,
    "Commercial Crime": Commercial_or_Financial,
    "Police Action": Police_Action,
    "Social Crime": Social_Crimes
}

In [18]:
def map_crime_group(crime):
    for group, crimes in crime_groups.items():
        if crime in crimes:
            return group
    return "Other"


In [19]:
merged["Crime_Group"] = merged["Crime_Type"].apply(map_crime_group)
merged.head()

,Station_name,District,Crime_Type,Crime_Count,National_placement,Provincial_placement,Count_direction,CREATE_DT,VERSION,geometry,Crime_Group
0,ABERDEEN,Sarah Baartman District,Murder,2.0,595.0,109.0,Stabilized,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",Violent Crime
1,ABERDEEN,Sarah Baartman District,Attempted murder,0.0,864.0,128.0,Stabilized,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",Violent Crime
2,ABERDEEN,Sarah Baartman District,Culpable homicide,1.0,830.0,120.0,Decreased,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",Violent Crime
3,ABERDEEN,Sarah Baartman District,Robbery with aggravating circumstances,0.0,1141.0,191.0,Decreased,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",Violent Crime
4,ABERDEEN,Sarah Baartman District,Common robbery,2.0,664.0,77.0,Stabilized,20251205,1.4.2,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",Violent Crime


In [20]:
grouped = merged.groupby(
    ["Station_name", "Crime_Group", "geometry"]
)["Crime_Count"].sum().reset_index()
grouped.head()


,Station_name,Crime_Group,geometry,Crime_Count
0,ABERDEEN,Commercial Crime,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",6.0
1,ABERDEEN,Crimes Against Children,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",0.0
2,ABERDEEN,Police Action,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",1.0
3,ABERDEEN,Property Damage,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",4.0
4,ABERDEEN,Sexual Offences,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359...",1.0


In [17]:
# To check If I made any spelling mistakes in any of the lists
filtered = merged[merged["Crime_Group"] == "Other"]
filtered.head()

,Station_name,District,Crime_Type,Crime_Count,National_placement,Provincial_placement,Count_direction,CREATE_DT,VERSION,geometry,Crime_Group


### Create pivot table to count each crime type in each Station

In [21]:
pivot = grouped.pivot(index='Station_name', columns='Crime_Group', values='Crime_Count').fillna(0)
pivot.head()

Crime_Group,Commercial Crime,Crimes Against Children,Police Action,Property Damage,Sexual Offences,Social Crime,Theft and Burglary,Violent Crime
Station_name,,,,,,,,
ABERDEEN,6.0,0.0,1.0,4.0,1.0,1.0,39.0,27.0
ACORNHOEK,47.0,3.0,17.0,39.0,34.0,1.0,141.0,229.0
ACTONVILLE,7.0,6.0,43.0,13.0,7.0,0.0,55.0,109.0
ADDO,6.0,0.0,2.0,29.0,4.0,3.0,95.0,81.0
ADELAIDE,6.0,0.0,0.0,6.0,9.0,3.0,54.0,55.0


### Station Polygons

In [22]:
geom = grouped.drop_duplicates('Station_name')[['Station_name', 'geometry']].set_index('Station_name')
geom.head()

,geometry
Station_name,
ABERDEEN,"POLYGON ((23.33305 -32.35114, 23.34384 -32.359..."
ACORNHOEK,"POLYGON ((31.16806 -24.56062, 31.16863 -24.605..."
ACTONVILLE,"POLYGON ((28.31868 -26.21993, 28.31871 -26.220..."
ADDO,"POLYGON ((25.69214 -33.38208, 25.7017 -33.4107..."
ADELAIDE,"POLYGON ((26.30944 -32.37606, 26.32127 -32.414..."


### Join Polygons to the pivot table

In [23]:
gdf = pivot.join(geom).reset_index()
gdf = gpd.GeoDataFrame(gdf, geometry='geometry')
gdf.crs = 'EPSG:4326'
sample_gdf = gdf.sample(100, random_state=42)
sample_gdf.head()

,Station_name,Commercial Crime,Crimes Against Children,Police Action,Property Damage,Sexual Offences,Social Crime,Theft and Burglary,Violent Crime,geometry
626,MASEMOLA,2.0,1.0,11.0,13.0,7.0,7.0,49.0,89.0,"POLYGON ((29.43383 -24.70969, 29.43385 -24.709..."
220,DORSET,3.0,0.0,0.0,1.0,0.0,0.0,6.0,1.0,"POLYGON ((28.43441 -23.84775, 28.43612 -23.849..."
678,MOKOPANE,59.0,1.0,105.0,26.0,15.0,7.0,190.0,130.0,"POLYGON ((29.12 -23.99646, 29.14463 -24.00753,..."
930,SCHOEMANSDAL,14.0,1.0,5.0,14.0,23.0,0.0,64.0,110.0,"POLYGON ((31.67019 -25.75687, 31.63512 -25.801..."
174,COOKHOUSE,4.0,1.0,3.0,7.0,1.0,0.0,41.0,39.0,"POLYGON ((25.81661 -32.48418, 25.81625 -32.485..."


In [ ]:
import folium
from folium import Choropleth, Circle, Marker, GeoJson
from folium.plugins import HeatMap, MarkerCluster, FeatureGroupSubGroup

m = folium.Map(location=[-33.92,18.84],  zoom_start=10)

In [ ]:
def get_color(value, max_val):
    if value == 0:
        return '#ffffff'
    intensity = min(value / max_val, 1.0)
    r = 255
    g = int(200 * (1 - intensity))
    b = int(100 * (1 - intensity))
    return f'#{r:02x}{g:02x}{b:02x}'

In [ ]:
crime_groups = [col for col in sample_gdf.columns if col not in ['Station_name','geometry']]
max_counts = {group: sample_gdf[group].max() for group in crime_groups}
print(crime_groups)
print(max_counts)

In [ ]:
geojson_data = sample_gdf.to_json()

for i, group in enumerate(crime_groups):
    max_val=max_counts[group]
    feature_group = folium.FeatureGroup(name=group, show=(i == 0))

    def style_function(feature, group=group, max_val=max_val):
        # station = feature['properties']['Station_name']
        count = feature['properties'].get(group, 0)
        color = get_color(count, max_val)
        return {
            'fillColor': color,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.7
        }
    
    folium.GeoJson(
        geojson_data, 
        style_function=style_function,
        tooltip=folium.GeoJsonTooltip(
            fields=['Station_name'] + crime_groups,
            aliases=['Station:'] + [f'{g}:' for g in crime_groups],
            localize=True
        )
    ).add_to(feature_group)
    
    feature_group.add_to(m)


#### Adding Legend

In [ ]:
legend_html = """
<div style="
position: fixed; 
bottom: 50px; left: 50px; width: 200px; height: 150px; 
background-color: white; 
border:2px solid grey; z-index:9999; font-size:14px;
padding: 10px;
">
<b>Crime Intensity</b><br>
Low &nbsp; <span style="background:#ffcccc;">&nbsp;&nbsp;&nbsp;</span><br>
Medium &nbsp; <span style="background:#ff6666;">&nbsp;&nbsp;&nbsp;</span><br>
High &nbsp; <span style="background:#ff0000;">&nbsp;&nbsp;&nbsp;</span><br>
</div>
"""



In [ ]:

m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(m)
m.save("crime_map.html")
